# Classical ML — Driver-Based Split

## 1. Load HOG Features

Load previously extracted HOG features and their corresponding labels.

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

# Locate project directory
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Load training metadata
train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

# Load saved HOG features (64x48)
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_64x48_train.npy"
)

# Extract labels and driver IDs
y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Verify matching number of samples
assert len(X_train) == len(y_train) == len(groups_train)

print("Features:", X_train.shape)
print("Labels:", y_train.shape)
print("Drivers:", len(np.unique(groups_train)))

Features: (15732, 1260)
Labels: (15732,)
Drivers: 18


## 2. Logistic Regression — Cross-Validation

Evaluate Logistic Regression using 4-fold StratifiedGroupKFold.

Drivers must not overlap between training and validation folds.

In [15]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create 4-fold driver-independent cross-validation
cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Create preprocessing + classification pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

# Train and evaluate on each fold
scores = cross_validate(
    model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display results
print("Accuracy per fold:", scores["test_accuracy"])
print("Macro-F1 per fold:", scores["test_f1_macro"])

print(f"\nMean Accuracy: {scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.40660377 0.4363725  0.59825674 0.51720311]
Macro-F1 per fold: [0.4266527  0.44388998 0.58989271 0.52773054]

Mean Accuracy: 0.4896
Accuracy Std: 0.0746

Mean Macro-F1: 0.4970
Macro-F1 Std: 0.0658


In [7]:
# Save cross-validation results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame({
    "model": "Logistic Regression",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": scores["test_accuracy"],
    "macro_f1": scores["test_f1_macro"]
})

results_df.to_csv(
    results_dir / "logistic_regression_64x48.csv",
    index=False
)

print(results_df)

                 model resolution  fold  accuracy  macro_f1
0  Logistic Regression      64x48     1  0.406604  0.426653
1  Logistic Regression      64x48     2  0.436373  0.443890
2  Logistic Regression      64x48     3  0.598257  0.589893
3  Logistic Regression      64x48     4  0.517203  0.527731


## 3. Support Vector Machine (SVM) — Cross-Validation

Evaluate a linear SVM using HOG features and 4-fold
driver-independent cross-validation.

In [5]:
import numpy as np
import pandas as pd
import json
import time
import sys
import os
import platform
import sklearn

from pathlib import Path
from datetime import datetime, timezone

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, cross_validate


# ============================================================
# 1. LOAD ONLY 64x48 HOG FEATURES
# ============================================================

ROOT = Path("..")

train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

X_train = np.load(
    ROOT / "data/processed/driver_based/hog_64x48_train.npy",
    mmap_mode="r"
)

y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Verify correct resolution
assert X_train.shape == (15732, 1260)
assert len(X_train) == len(y_train) == len(groups_train)

print("HOG Features:", X_train.shape)
print("Unique Drivers:", len(np.unique(groups_train)))


# ============================================================
# 2. CREATE DRIVER-INDEPENDENT CROSS-VALIDATION
# ============================================================

cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

splits = list(cv.split(X_train, y_train, groups_train))

# Verify no driver overlap
for train_idx, val_idx in splits:
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    assert len(train_drivers & val_drivers) == 0


# ============================================================
# 3. CREATE OPTIMIZED LINEAR SVM
# ============================================================

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        dual=False,
        max_iter=1000,
        tol=1e-2,
        random_state=42
    ))
])


# ============================================================
# 4. TRAIN AND EVALUATE
# ============================================================

started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    cv=splits,
    scoring=["accuracy", "f1_macro"],
    n_jobs=4,
    verbose=2,
    return_estimator=True,
    error_score="raise"
)

total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()


# ============================================================
# 5. CHECK OPTIMIZATION ITERATIONS
# ============================================================

iterations = [
    int(model.named_steps["svm"].n_iter_)
    for model in svm_scores["estimator"]
]

iteration_limit_reached = [
    n >= 1000 for n in iterations
]


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("\nAccuracy per fold:", svm_scores["test_accuracy"])
print("Macro-F1 per fold:", svm_scores["test_f1_macro"])

print(f"\nMean Accuracy: {svm_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {svm_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {svm_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {svm_scores['test_f1_macro'].std():.4f}")

print(f"\nTotal execution time: {total_time:.2f} seconds")
print("Iterations per fold:", iterations)
print("Reached iteration limit:", iteration_limit_reached)


# ============================================================
# 7. SAVE FOLD RESULTS TO CSV
# ============================================================

results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"],
    "score_time": svm_scores["score_time"],
    "iterations": iterations,
    "iteration_limit_reached": iteration_limit_reached
})

csv_path = results_dir / "linear_svm_64x48_primal.csv"

svm_results.to_csv(csv_path, index=False)


# ============================================================
# 8. RECORD DETAILS FOR EVERY FOLD
# ============================================================

fold_details = []

for fold, (train_idx, val_idx) in enumerate(splits, start=1):

    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    fold_details.append({
        "fold": fold,

        "training_images": int(len(train_idx)),
        "validation_images": int(len(val_idx)),

        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),

        "driver_overlap": len(train_drivers & val_drivers),

        "accuracy": float(
            svm_scores["test_accuracy"][fold - 1]
        ),

        "macro_f1": float(
            svm_scores["test_f1_macro"][fold - 1]
        ),

        "fit_time_seconds": float(
            svm_scores["fit_time"][fold - 1]
        ),

        "score_time_seconds": float(
            svm_scores["score_time"][fold - 1]
        ),

        "iterations": iterations[fold - 1],
        "iteration_limit_reached": iteration_limit_reached[fold - 1]
    })


# ============================================================
# 9. CREATE COMPLETE JSON METADATA
# ============================================================

metadata = {
    "experiment": {
        "model": "Linear SVM",
        "resolution": "64x48",
        "feature_type": "HOG",
        "experiment_variant": "primal",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_64x48_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "LinearSVC"],

        "scaler_parameters":
            svm_model.named_steps["scaler"].get_params(),

        "svm_parameters":
            svm_model.named_steps["svm"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "parallel_workers": 4,
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(
            svm_scores["test_accuracy"].mean()
        ),

        "std_accuracy": float(
            svm_scores["test_accuracy"].std()
        ),

        "mean_macro_f1": float(
            svm_scores["test_f1_macro"].mean()
        ),

        "std_macro_f1": float(
            svm_scores["test_f1_macro"].std()
        )
    },

    "timing": {
        "total_execution_seconds": float(total_time),

        "total_fit_time_seconds": float(
            svm_scores["fit_time"].sum()
        ),

        "total_score_time_seconds": float(
            svm_scores["score_time"].sum()
        )
    },

    "convergence": {
        "iterations_per_fold": iterations,
        "iteration_limit_reached": iteration_limit_reached,
        "convergence_verified": False,
        "note": "Review worker output for convergence warnings."
    },

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": os.cpu_count()
    }
}


# ============================================================
# 10. SAVE JSON METADATA
# ============================================================

json_path = results_dir / "linear_svm_64x48_primal_metadata.json"

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)


# ============================================================
# 11. VERIFY SAVED RESULTS
# ============================================================

print("\nLinear SVM Results — 64x48")
print(svm_results)

print("\nCSV saved:", csv_path)
print("JSON saved:", json_path)

print("\nExperiment completed successfully!")

HOG Features: (15732, 1260)
Unique Drivers: 18


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END .................................................... total time=  36.5s
[CV] END .................................................... total time=  37.0s
[CV] END .................................................... total time=  39.6s
[CV] END .................................................... total time=  39.8s

Accuracy per fold: [0.39268868 0.38078986 0.51373481 0.45061043]
Macro-F1 per fold: [0.39801667 0.37860993 0.49147014 0.45083992]

Mean Accuracy: 0.4345
Accuracy Std: 0.0528

Mean Macro-F1: 0.4297
Macro-F1 Std: 0.0444

Total execution time: 40.33 seconds
Iterations per fold: [11, 11, 12, 11]
Reached iteration limit: [False, False, False, False]

Linear SVM Results — 64x48
        model resolution  fold  accuracy  macro_f1   fit_time  score_time  \
0  Linear SVM      64x48     1  0.392689  0.398017  37.008467    0.026051   
1  Linear SVM      64x48     2  0.380790  0.378610  36.428238    0.026351   
2  Linear SVM      64x48     3  0.513735  0.491470  39.588218    0.01

[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:   40.3s finished


In [9]:
# Save SVM cross-validation results
svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"]
})

svm_results.to_csv(
    results_dir / "linear_svm_64x48.csv",
    index=False
)

print(svm_results)

        model resolution  fold  accuracy  macro_f1    fit_time
0  Linear SVM      64x48     1  0.390566  0.389648  188.063229
1  Linear SVM      64x48     2  0.383471  0.384355  197.629248
2  Linear SVM      64x48     3  0.521395  0.498647  199.263910
3  Linear SVM      64x48     4  0.457547  0.460508  210.143842


## 4. Random Forest — Cross-Validation

Evaluate Random Forest using HOG features and 4-fold
driver-independent cross-validation.

In [19]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 decision trees
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Evaluate using the same driver-independent folds
rf_scores = cross_validate(
    rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display cross-validation results
print("Accuracy per fold:", rf_scores["test_accuracy"])
print("Macro-F1 per fold:", rf_scores["test_f1_macro"])

print(f"\nMean Accuracy: {rf_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {rf_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {rf_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {rf_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.40471698 0.49536811 0.62651875 0.59100999]
Macro-F1 per fold: [0.4007372  0.47444469 0.60233494 0.57636729]

Mean Accuracy: 0.5294
Accuracy Std: 0.0865

Mean Macro-F1: 0.5135
Macro-F1 Std: 0.0808


In [22]:
saved = pd.read_csv(
    results_dir / "random_forest_64x48.csv"
)

print(saved[["fold", "accuracy", "macro_f1"]])

   fold  accuracy  macro_f1
0     1  0.404717  0.400737
1     2  0.495368  0.474445
2     3  0.626519  0.602335
3     4  0.591010  0.576367


In [13]:
# Check training time for each fold
print("Training times:", rf_scores["fit_time"])

# Check validation scoring time
print("Scoring times:", rf_scores["score_time"])

# Check if any folds produced invalid results
print("Accuracy scores:", rf_scores["test_accuracy"])

Training times: [9.31164742 9.06690311 9.60424566 9.89199662]
Scoring times: [0.09394193 0.09374928 0.08465672 0.08376336]
Accuracy scores: [0.34245283 0.61774744 0.62810354 0.63346282]


In [21]:
import json

# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# Save results for each fold
rf_results = pd.DataFrame({
    "model": "Random Forest",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": rf_scores["test_accuracy"],
    "macro_f1": rf_scores["test_f1_macro"],
    "fit_time": rf_scores["fit_time"],
    "score_time": rf_scores["score_time"]
})

rf_results.to_csv(
    results_dir / "random_forest_64x48.csv",
    index=False
)

# Save experiment metadata
metadata = {
    "model": "Random Forest",
    "resolution": "64x48",
    "hog_features": 1260,
    "n_estimators": 500,
    "hyperparameters": rf_model.get_params(),
    "cv_method": "StratifiedGroupKFold",
    "n_splits": 4,
    "random_state_cv": 42,
    "mean_accuracy": float(rf_scores["test_accuracy"].mean()),
    "std_accuracy": float(rf_scores["test_accuracy"].std()),
    "mean_macro_f1": float(rf_scores["test_f1_macro"].mean()),
    "std_macro_f1": float(rf_scores["test_f1_macro"].std()),
    "notebook_elapsed_seconds_approx": 17.1
}

with open(results_dir / "random_forest_64x48_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Results and metadata saved!")
print("Random Forest n_jobs:", rf_model.get_params()["n_jobs"])

Results and metadata saved!
Random Forest n_jobs: -1


## 5. K-Nearest Neighbors (KNN) — Cross-Validation

Evaluate KNN using HOG features and 4-fold
driver-independent cross-validation.

In [16]:
from sklearn.neighbors import KNeighborsClassifier

# Create KNN pipeline with feature scaling
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        n_jobs=-1
    ))
])

# Evaluate using the same driver-independent folds
knn_scores = cross_validate(
    knn_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display cross-validation results
print("Accuracy per fold:", knn_scores["test_accuracy"])
print("Macro-F1 per fold:", knn_scores["test_f1_macro"])

print(f"\nMean Accuracy: {knn_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {knn_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {knn_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {knn_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.34716981 0.39980497 0.54120444 0.49556049]
Macro-F1 per fold: [0.33596405 0.38707672 0.53285261 0.48806661]

Mean Accuracy: 0.4459
Accuracy Std: 0.0765

Mean Macro-F1: 0.4360
Macro-F1 Std: 0.0782


In [15]:
# Save KNN cross-validation results
knn_results = pd.DataFrame({
    "model": "KNN",
    "resolution": "64x48",
    "fold": range(1, 5),
    "accuracy": knn_scores["test_accuracy"],
    "macro_f1": knn_scores["test_f1_macro"],
    "fit_time": knn_scores["fit_time"],
    "score_time": knn_scores["score_time"]
})

knn_results.to_csv(
    results_dir / "knn_64x48.csv",
    index=False
)

print(knn_results)

  model resolution  fold  accuracy  macro_f1  fit_time  score_time
0   KNN      64x48     1  0.347170  0.335964  0.070482    0.285926
1   KNN      64x48     2  0.399805  0.387077  0.065658    0.234601
2   KNN      64x48     3  0.541204  0.532853  0.066580    0.213542
3   KNN      64x48     4  0.495560  0.488067  0.065749    0.219081


## 6. HOG Resolution Comparison

Investigate how image resolution affects classification performance.

Resolutions:
- 64×48
- 128×96
- 256×192

Use the same model and cross-validation folds for fair comparison.

# Classical Machine Learning — Driver-Based Split

## 1. HOG Features: 128×96

Load and verify the training features and labels.

Each image is represented by 5940 HOG features.
The final test set remains untouched.

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..")

# Load training metadata
train_df = pd.read_csv(ROOT / "data/splits/driver_based/train.csv")

# Load ONLY 128x96 HOG features
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_128x96_train.npy",
    mmap_mode="r"
)

# Load labels and driver IDs
y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Check dimensions
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Unique drivers:", len(np.unique(groups_train)))
print("Classes:", np.unique(y_train))

X_train: (15732, 5940)
y_train: (15732,)
Unique drivers: 18
Classes: ['c0' 'c1' 'c2' 'c3' 'c4' 'c5' 'c6' 'c7' 'c8' 'c9']


## 2. Logistic Regression — 4-Fold Cross-Validation

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Create Logistic Regression pipeline
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])


# Train and evaluate using existing 4-fold CV
lr_scores = cross_validate(
    lr_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1
)

# Display results for all 4 folds
print("Accuracy per fold:", lr_scores["test_accuracy"])
print("Macro-F1 per fold:", lr_scores["test_f1_macro"])

print(f"\nMean Accuracy: {lr_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {lr_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {lr_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {lr_scores['test_f1_macro'].std():.4f}")

Accuracy per fold: [0.41014151 0.58069235 0.76069731 0.69034406]
Macro-F1 per fold: [0.4255203  0.57923797 0.75273559 0.68602231]

Mean Accuracy: 0.6105
Accuracy Std: 0.1323

Mean Macro-F1: 0.6109
Macro-F1 Std: 0.1236


In [7]:
# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# Organize results from all 4 folds
lr_results = pd.DataFrame({
    "model": "Logistic Regression",
    "resolution": "128x96",
    "fold": range(1, 5),
    "accuracy": lr_scores["test_accuracy"],
    "macro_f1": lr_scores["test_f1_macro"],
    "fit_time": lr_scores["fit_time"]
})

# Save results
lr_results.to_csv(
    results_dir / "logistic_regression_128x96.csv",
    index=False
)

print(lr_results)

                 model resolution  fold  accuracy  macro_f1  fit_time
0  Logistic Regression     128x96     1  0.410142  0.425520  3.298761
1  Logistic Regression     128x96     2  0.580692  0.579238  3.929901
2  Logistic Regression     128x96     3  0.760697  0.752736  3.775546
3  Logistic Regression     128x96     4  0.690344  0.686022  3.634526


## 3. Linear SVM — 4-Fold Cross-Validation

In [8]:
import numpy as np
import pandas as pd
import json
import time
import sys
import os
import platform
import sklearn

from pathlib import Path
from datetime import datetime, timezone

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, cross_validate


# ============================================================
# 1. LOAD ONLY 128x96 HOG FEATURES
# ============================================================

ROOT = Path("..")

train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

X_train = np.load(
    ROOT / "data/processed/driver_based/hog_128x96_train.npy",
    mmap_mode="r"
)

y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Verify correct resolution
assert X_train.shape == (15732, 5940)
assert len(X_train) == len(y_train) == len(groups_train)

print("HOG Features:", X_train.shape)
print("Unique Drivers:", len(np.unique(groups_train)))


# ============================================================
# 2. CREATE DRIVER-INDEPENDENT CROSS-VALIDATION
# ============================================================

cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

splits = list(cv.split(X_train, y_train, groups_train))

# Verify no driver overlap
for train_idx, val_idx in splits:
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    assert len(train_drivers & val_drivers) == 0


# ============================================================
# 3. CREATE OPTIMIZED LINEAR SVM
# ============================================================

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        dual=False,
        max_iter=1000,
        tol=1e-2,
        random_state=42
    ))
])


# ============================================================
# 4. TRAIN AND EVALUATE
# ============================================================

started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    cv=splits,
    scoring=["accuracy", "f1_macro"],
    n_jobs=4,
    verbose=2,
    return_estimator=True,
    error_score="raise"
)

total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()


# ============================================================
# 5. CHECK OPTIMIZATION ITERATIONS
# ============================================================

iterations = [
    int(model.named_steps["svm"].n_iter_)
    for model in svm_scores["estimator"]
]

iteration_limit_reached = [
    n >= 1000 for n in iterations
]


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("\nAccuracy per fold:", svm_scores["test_accuracy"])
print("Macro-F1 per fold:", svm_scores["test_f1_macro"])

print(f"\nMean Accuracy: {svm_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {svm_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {svm_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {svm_scores['test_f1_macro'].std():.4f}")

print(f"\nTotal execution time: {total_time:.2f} seconds")
print("Iterations per fold:", iterations)
print("Reached iteration limit:", iteration_limit_reached)


# ============================================================
# 7. SAVE FOLD RESULTS TO CSV
# ============================================================

results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "128x96",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"],
    "score_time": svm_scores["score_time"],
    "iterations": iterations,
    "iteration_limit_reached": iteration_limit_reached
})

csv_path = results_dir / "linear_svm_128x96_primal.csv"

svm_results.to_csv(csv_path, index=False)


# ============================================================
# 8. RECORD DETAILS FOR EVERY FOLD
# ============================================================

fold_details = []

for fold, (train_idx, val_idx) in enumerate(splits, start=1):

    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    fold_details.append({
        "fold": fold,

        "training_images": int(len(train_idx)),
        "validation_images": int(len(val_idx)),

        "training_drivers": sorted(map(str, train_drivers)),
        "validation_drivers": sorted(map(str, val_drivers)),

        "driver_overlap": len(train_drivers & val_drivers),

        "accuracy": float(
            svm_scores["test_accuracy"][fold - 1]
        ),

        "macro_f1": float(
            svm_scores["test_f1_macro"][fold - 1]
        ),

        "fit_time_seconds": float(
            svm_scores["fit_time"][fold - 1]
        ),

        "score_time_seconds": float(
            svm_scores["score_time"][fold - 1]
        ),

        "iterations": iterations[fold - 1],
        "iteration_limit_reached": iteration_limit_reached[fold - 1]
    })


# ============================================================
# 9. CREATE COMPLETE JSON METADATA
# ============================================================

metadata = {
    "experiment": {
        "model": "Linear SVM",
        "resolution": "128x96",
        "feature_type": "HOG",
        "experiment_variant": "primal",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_128x96_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "LinearSVC"],

        "scaler_parameters":
            svm_model.named_steps["scaler"].get_params(),

        "svm_parameters":
            svm_model.named_steps["svm"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "parallel_workers": 4,
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(
            svm_scores["test_accuracy"].mean()
        ),

        "std_accuracy": float(
            svm_scores["test_accuracy"].std()
        ),

        "mean_macro_f1": float(
            svm_scores["test_f1_macro"].mean()
        ),

        "std_macro_f1": float(
            svm_scores["test_f1_macro"].std()
        )
    },

    "timing": {
        "total_execution_seconds": float(total_time),

        "total_fit_time_seconds": float(
            svm_scores["fit_time"].sum()
        ),

        "total_score_time_seconds": float(
            svm_scores["score_time"].sum()
        )
    },

    "convergence": {
        "iterations_per_fold": iterations,
        "iteration_limit_reached": iteration_limit_reached,
        "convergence_verified": False,
        "note": "Review worker output for convergence warnings."
    },

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": os.cpu_count()
    }
}


# ============================================================
# 10. SAVE JSON METADATA
# ============================================================

json_path = results_dir / "linear_svm_128x96_primal_metadata.json"

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)


# ============================================================
# 11. VERIFY SAVED RESULTS
# ============================================================

print("\nLinear SVM Results — 128x96")
print(svm_results)

print("\nCSV saved:", csv_path)
print("JSON saved:", json_path)

print("\nExperiment completed successfully!")

HOG Features: (15732, 5940)
Unique Drivers: 18


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END .................................................... total time= 2.4min
[CV] END .................................................... total time= 2.5min
[CV] END .................................................... total time= 2.6min
[CV] END .................................................... total time= 2.6min

Accuracy per fold: [0.37735849 0.45148708 0.620972   0.55493896]
Macro-F1 per fold: [0.37690471 0.44884559 0.60941378 0.53838909]

Mean Accuracy: 0.5012
Accuracy Std: 0.0936

Mean Macro-F1: 0.4934
Macro-F1 Std: 0.0881

Total execution time: 157.44 seconds
Iterations per fold: [6, 5, 5, 5]
Reached iteration limit: [False, False, False, False]

Linear SVM Results — 128x96
        model resolution  fold  accuracy  macro_f1    fit_time  score_time  \
0  Linear SVM     128x96     1  0.377358  0.376905  141.629436    0.119692   
1  Linear SVM     128x96     2  0.451487  0.448846  147.288656    0.084089   
2  Linear SVM     128x96     3  0.620972  0.609414  156.872332    0.

[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:  2.6min finished


## 4. Random Forest — 4-Fold Cross-Validation

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 trees
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Evaluate using the same driver-independent folds
rf_scores = cross_validate(
    rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1,
    verbose=2
)

# Display results
print("Accuracy per fold:", rf_scores["test_accuracy"])
print("Macro-F1 per fold:", rf_scores["test_f1_macro"])

print(f"\nMean Accuracy: {rf_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {rf_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {rf_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {rf_scores['test_f1_macro'].std():.4f}")

[CV] END .................................................... total time=   9.4s
[CV] END .................................................... total time=   9.1s
[CV] END .................................................... total time=   9.4s
[CV] END .................................................... total time=   9.8s
Accuracy per fold: [0.34245283 0.61774744 0.62810354 0.63346282]
Macro-F1 per fold: [0.34124818 0.59207357 0.60380973 0.61935264]

Mean Accuracy: 0.5554
Accuracy Std: 0.1231

Mean Macro-F1: 0.5391
Macro-F1 Std: 0.1147


[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:   37.7s finished


In [24]:
# Load ONLY 128x96 HOG features
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_128x96_train.npy",
    mmap_mode="r"
)

# Verify dimensions
assert X_train.shape == (15732, 5940)

print("Current HOG features:", X_train.shape)

Current HOG features: (15732, 5940)


## 5. K-Nearest Neighbors (KNN) — 4-Fold Cross-Validation

In [25]:
import time
from datetime import datetime
from sklearn.neighbors import KNeighborsClassifier

# Create KNN pipeline
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        n_jobs=-1
    ))
])

# Start timer
start_datetime = datetime.now().astimezone().isoformat()
start_time = time.perf_counter()

# 4-fold driver-independent cross-validation
knn_scores = cross_validate(
    knn_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=1,
    verbose=2
)

# Record total execution time
total_time = time.perf_counter() - start_time
end_datetime = datetime.now().astimezone().isoformat()

# Display results
print("Accuracy per fold:", knn_scores["test_accuracy"])
print("Macro-F1 per fold:", knn_scores["test_f1_macro"])

print(f"\nMean Accuracy: {knn_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {knn_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {knn_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {knn_scores['test_f1_macro'].std():.4f}")

print(f"\nTotal execution time: {total_time:.2f} seconds")

[CV] END .................................................... total time=   1.7s
[CV] END .................................................... total time=   1.6s
[CV] END .................................................... total time=   1.5s
[CV] END .................................................... total time=   1.5s
Accuracy per fold: [0.4740566  0.50633837 0.58267301 0.57241953]
Macro-F1 per fold: [0.45366837 0.48134506 0.5666081  0.56655105]

Mean Accuracy: 0.5339
Accuracy Std: 0.0453

Mean Macro-F1: 0.5170
Macro-F1 Std: 0.0505

Total execution time: 6.31 seconds


[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    6.3s finished


In [27]:
print(X_train.shape)

(15732, 5940)


In [26]:
import json
import sys
import platform
import sklearn

# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# 1. Save individual fold results
knn_results = pd.DataFrame({
    "model": "KNN",
    "resolution": "128x96",
    "fold": range(1, 5),
    "accuracy": knn_scores["test_accuracy"],
    "macro_f1": knn_scores["test_f1_macro"],
    "fit_time": knn_scores["fit_time"],
    "score_time": knn_scores["score_time"]
})

knn_results.to_csv(
    results_dir / "knn_128x96.csv",
    index=False
)

# 2. Record cross-validation fold information
fold_details = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups_train), start=1
):
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    fold_details.append({
        "fold": fold,
        "training_images": len(train_idx),
        "validation_images": len(val_idx),
        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),
        "driver_overlap": len(train_drivers & val_drivers),
        "accuracy": float(knn_scores["test_accuracy"][fold - 1]),
        "macro_f1": float(knn_scores["test_f1_macro"][fold - 1]),
        "fit_time_seconds": float(knn_scores["fit_time"][fold - 1]),
        "score_time_seconds": float(knn_scores["score_time"][fold - 1])
    })

# 3. Complete experiment metadata
metadata = {
    "experiment": {
        "model": "KNN",
        "resolution": "128x96",
        "feature_type": "HOG",
        "started_at": start_datetime,
        "completed_at": end_datetime
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_128x96_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "KNeighborsClassifier"],
        "scaler_parameters": knn_model.named_steps["scaler"].get_params(),
        "knn_parameters": knn_model.named_steps["knn"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(knn_scores["test_accuracy"].mean()),
        "std_accuracy": float(knn_scores["test_accuracy"].std()),
        "mean_macro_f1": float(knn_scores["test_f1_macro"].mean()),
        "std_macro_f1": float(knn_scores["test_f1_macro"].std())
    },

    "timing": {
        "total_execution_seconds": float(total_time),
        "total_fit_time_seconds": float(knn_scores["fit_time"].sum()),
        "total_score_time_seconds": float(knn_scores["score_time"].sum())
    },

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu": platform.processor()
    }
}

# 4. Save metadata as JSON
with open(results_dir / "knn_128x96_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("KNN results and metadata saved successfully!")
print(f"Total execution time: {total_time:.2f} seconds")
print(knn_results)

KNN results and metadata saved successfully!
Total execution time: 6.31 seconds
  model resolution  fold  accuracy  macro_f1  fit_time  score_time
0   KNN     128x96     1  0.474057  0.453668  0.291848    1.393511
1   KNN     128x96     2  0.506338  0.481345  0.284413    1.326039
2   KNN     128x96     3  0.582673  0.566608  0.288630    1.195044
3   KNN     128x96     4  0.572420  0.566551  0.288223    1.217745


# Experiment 3: Classical ML — HOG at 256×192

In [28]:
# Load ONLY 256x192 HOG features
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_256x192_train.npy",
    mmap_mode="r"
)

# Verify dimensions
print("Features:", X_train.shape)
print("Labels:", y_train.shape)
print("Unique drivers:", len(np.unique(groups_train)))

# Verify data consistency
assert X_train.shape == (15732, 25668)
assert len(X_train) == len(y_train) == len(groups_train)

print("256x192 dataset ready!")

Features: (15732, 25668)
Labels: (15732,)
Unique drivers: 18
256x192 dataset ready!


## 2. Logistic Regression — 4-Fold Cross-Validation

In [29]:
import time
import json
import sys
import platform
import warnings
import sklearn
import numpy as np
import pandas as pd

from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

# Verify the correct HOG resolution
assert X_train.shape == (15732, 25668), "Incorrect HOG resolution!"

# Create Logistic Regression pipeline
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

# Start timer
started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

# Train using 4-fold driver-independent CV
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")

    lr_scores = cross_validate(
        lr_model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=1,
        verbose=2
    )

# Record execution time
total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()

# Display results
print("\nAccuracy per fold:", lr_scores["test_accuracy"])
print("Macro-F1 per fold:", lr_scores["test_f1_macro"])

print(f"\nMean Accuracy: {lr_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {lr_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {lr_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {lr_scores['test_f1_macro'].std():.4f}")

print(f"\nTotal execution time: {total_time:.2f} seconds")

# Create results directory
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

# Save individual fold results
lr_results = pd.DataFrame({
    "model": "Logistic Regression",
    "resolution": "256x192",
    "fold": range(1, 5),
    "accuracy": lr_scores["test_accuracy"],
    "macro_f1": lr_scores["test_f1_macro"],
    "fit_time": lr_scores["fit_time"],
    "score_time": lr_scores["score_time"]
})

lr_results.to_csv(
    results_dir / "logistic_regression_256x192.csv",
    index=False
)

# Record information about each CV fold
fold_details = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups_train), start=1
):
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    fold_details.append({
        "fold": fold,
        "training_images": len(train_idx),
        "validation_images": len(val_idx),
        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),
        "driver_overlap": len(train_drivers & val_drivers),
        "accuracy": float(lr_scores["test_accuracy"][fold - 1]),
        "macro_f1": float(lr_scores["test_f1_macro"][fold - 1]),
        "fit_time_seconds": float(lr_scores["fit_time"][fold - 1]),
        "score_time_seconds": float(lr_scores["score_time"][fold - 1])
    })

# Save complete metadata
metadata = {
    "experiment": {
        "model": "Logistic Regression",
        "resolution": "256x192",
        "feature_type": "HOG",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_256x192_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "LogisticRegression"],
        "scaler_parameters": lr_model.named_steps["scaler"].get_params(),
        "logistic_regression_parameters": lr_model.named_steps["logreg"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(lr_scores["test_accuracy"].mean()),
        "std_accuracy": float(lr_scores["test_accuracy"].std()),
        "mean_macro_f1": float(lr_scores["test_f1_macro"].mean()),
        "std_macro_f1": float(lr_scores["test_f1_macro"].std())
    },

    "timing": {
        "total_execution_seconds": float(total_time),
        "total_fit_time_seconds": float(lr_scores["fit_time"].sum()),
        "total_score_time_seconds": float(lr_scores["score_time"].sum())
    },

    "warnings": [
        {
            "category": w.category.__name__,
            "message": str(w.message)
        }
        for w in caught_warnings
    ],

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": __import__("os").cpu_count()
    }
}

# Save metadata as JSON
with open(
    results_dir / "logistic_regression_256x192_metadata.json", "w"
) as f:
    json.dump(metadata, f, indent=4)

print("\nExperiment results and metadata saved successfully!")
print(lr_results)

if caught_warnings:
    print(f"\nTraining generated {len(caught_warnings)} warning(s).")

[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   5.0s
[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   5.1s

Accuracy per fold: [0.43018868 0.53339834 0.75118859 0.64150943]
Macro-F1 per fold: [0.44168944 0.53075295 0.7455961  0.62860474]

Mean Accuracy: 0.5891
Accuracy Std: 0.1198

Mean Macro-F1: 0.5867
Macro-F1 Std: 0.1131

Total execution time: 19.73 seconds

Experiment results and metadata saved successfully!
                 model resolution  fold  accuracy  macro_f1  fit_time  \
0  Logistic Regression    256x192     1  0.430189  0.441689  4.668184   
1  Logistic Regression    256x192     2  0.533398  0.530753  4.839540   
2  Logistic Regression    256x192     3  0.751189  0.745596  4.645575   
3  Logistic Regression    256x192     4  0.641509  0.628605  4.966542   

 

[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:   19.7s finished


## 4. Random Forest — 4-Fold Cross-Validation

In [30]:
import time
import json
import sys
import platform
import sklearn
import warnings

from datetime import datetime, timezone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

# 1. Verify resolution
assert X_train.shape == (15732, 25668), "Incorrect HOG resolution!"

# 2. Create Random Forest
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# 3. Start timer
started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

# 4. Run 4-fold driver-independent cross-validation
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")

    rf_scores = cross_validate(
        rf_model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=1,
        verbose=2
    )

# 5. Record execution time
total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()

# 6. Save individual fold results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

rf_results = pd.DataFrame({
    "model": "Random Forest",
    "resolution": "256x192",
    "fold": range(1, 5),
    "accuracy": rf_scores["test_accuracy"],
    "macro_f1": rf_scores["test_f1_macro"],
    "fit_time": rf_scores["fit_time"],
    "score_time": rf_scores["score_time"]
})

rf_results.to_csv(
    results_dir / "random_forest_256x192.csv",
    index=False
)

# 7. Record complete fold details
fold_details = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups_train), start=1
):
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    assert not (train_drivers & val_drivers)

    fold_details.append({
        "fold": fold,
        "training_images": int(len(train_idx)),
        "validation_images": int(len(val_idx)),
        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),
        "accuracy": float(rf_scores["test_accuracy"][fold-1]),
        "macro_f1": float(rf_scores["test_f1_macro"][fold-1]),
        "fit_time_seconds": float(rf_scores["fit_time"][fold-1]),
        "score_time_seconds": float(rf_scores["score_time"][fold-1])
    })

# 8. Complete experiment metadata
metadata = {
    "experiment": {
        "model": "Random Forest",
        "resolution": "256x192",
        "feature_type": "HOG",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_256x192_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "hyperparameters": rf_model.get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "parallel_workers": 1,
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(rf_scores["test_accuracy"].mean()),
        "std_accuracy": float(rf_scores["test_accuracy"].std()),
        "mean_macro_f1": float(rf_scores["test_f1_macro"].mean()),
        "std_macro_f1": float(rf_scores["test_f1_macro"].std())
    },

    "timing": {
        "total_execution_seconds": float(total_time),
        "total_fit_time_seconds": float(rf_scores["fit_time"].sum()),
        "total_score_time_seconds": float(rf_scores["score_time"].sum())
    },

    "warnings": [
        {
            "category": w.category.__name__,
            "message": str(w.message)
        }
        for w in caught_warnings
    ],

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": __import__("os").cpu_count()
    }
}

# 9. Save metadata
with open(
    results_dir / "random_forest_256x192_metadata.json", "w"
) as f:
    json.dump(metadata, f, indent=4)

# 10. Display results
print("\nRandom Forest Results — 256x192")
print(rf_results)

print(f"\nMean Accuracy: {rf_scores['test_accuracy'].mean():.4f}")
print(f"Mean Macro-F1: {rf_scores['test_f1_macro'].mean():.4f}")
print(f"Total execution time: {total_time:.2f} seconds")

print("\nCSV and JSON saved successfully!")

[CV] END .................................................... total time=  20.1s
[CV] END .................................................... total time=  19.6s
[CV] END .................................................... total time=  21.0s
[CV] END .................................................... total time=  21.1s

Random Forest Results — 256x192
           model resolution  fold  accuracy  macro_f1   fit_time  score_time
0  Random Forest    256x192     1  0.344575  0.341709  20.040015    0.104469
1  Random Forest    256x192     2  0.556802  0.527583  19.489048    0.114733
2  Random Forest    256x192     3  0.636027  0.612748  20.866510    0.102995
3  Random Forest    256x192     4  0.592952  0.576308  21.033589    0.103371

Mean Accuracy: 0.5326
Mean Macro-F1: 0.5146
Total execution time: 81.88 seconds

CSV and JSON saved successfully!


[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:  1.4min finished


## 5. K-Nearest Neighbors (KNN) — 4-Fold Cross-Validation

In [31]:
import time
import json
import sys
import platform
import warnings
import sklearn
import numpy as np
import pandas as pd

from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate

# 1. Verify dataset
assert X_train.shape == (15732, 25668), "Incorrect HOG resolution!"
assert len(X_train) == len(y_train) == len(groups_train)

# 2. Create KNN pipeline
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        n_jobs=-1
    ))
])

# 3. Start timer
started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

# 4. Perform driver-independent cross-validation
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")

    knn_scores = cross_validate(
        knn_model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=1,
        verbose=2
    )

# 5. Record execution time
total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()

# 6. Save per-fold results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

knn_results = pd.DataFrame({
    "model": "KNN",
    "resolution": "256x192",
    "fold": range(1, 5),
    "accuracy": knn_scores["test_accuracy"],
    "macro_f1": knn_scores["test_f1_macro"],
    "fit_time": knn_scores["fit_time"],
    "score_time": knn_scores["score_time"]
})

knn_results.to_csv(
    results_dir / "knn_256x192.csv",
    index=False
)

# 7. Record cross-validation fold information
fold_details = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups_train), start=1
):
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    assert not (train_drivers & val_drivers)

    fold_details.append({
        "fold": fold,
        "training_images": int(len(train_idx)),
        "validation_images": int(len(val_idx)),
        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),
        "driver_overlap": len(train_drivers & val_drivers),
        "accuracy": float(knn_scores["test_accuracy"][fold-1]),
        "macro_f1": float(knn_scores["test_f1_macro"][fold-1]),
        "fit_time_seconds": float(knn_scores["fit_time"][fold-1]),
        "score_time_seconds": float(knn_scores["score_time"][fold-1])
    })

# 8. Save complete experiment metadata
metadata = {
    "experiment": {
        "model": "KNN",
        "resolution": "256x192",
        "feature_type": "HOG",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_256x192_train.npy",
        "metadata_file": "data/splits/driver_based/train.csv",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "KNeighborsClassifier"],
        "scaler_parameters": knn_model.named_steps["scaler"].get_params(),
        "knn_parameters": knn_model.named_steps["knn"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "parallel_workers": 1,
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(knn_scores["test_accuracy"].mean()),
        "std_accuracy": float(knn_scores["test_accuracy"].std()),
        "mean_macro_f1": float(knn_scores["test_f1_macro"].mean()),
        "std_macro_f1": float(knn_scores["test_f1_macro"].std())
    },

    "timing": {
        "total_execution_seconds": float(total_time),
        "total_fit_time_seconds": float(knn_scores["fit_time"].sum()),
        "total_score_time_seconds": float(knn_scores["score_time"].sum())
    },

    "warnings": [
        {
            "category": w.category.__name__,
            "message": str(w.message)
        }
        for w in caught_warnings
    ],

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": __import__("os").cpu_count()
    }
}

# 9. Save JSON metadata
with open(
    results_dir / "knn_256x192_metadata.json", "w"
) as f:
    json.dump(metadata, f, indent=4)

# 10. Display results
print("\nKNN Results — 256x192")
print(knn_results)

print(f"\nMean Accuracy: {knn_scores['test_accuracy'].mean():.4f}")
print(f"Accuracy Std: {knn_scores['test_accuracy'].std():.4f}")

print(f"\nMean Macro-F1: {knn_scores['test_f1_macro'].mean():.4f}")
print(f"Macro-F1 Std: {knn_scores['test_f1_macro'].std():.4f}")

print(f"\nTotal execution time: {total_time:.2f} seconds")

print("\nCSV and JSON saved successfully!")

[CV] END .................................................... total time=   6.8s
[CV] END .................................................... total time=   6.5s
[CV] END .................................................... total time=   6.1s
[CV] END .................................................... total time=   6.1s

KNN Results — 256x192
  model resolution  fold  accuracy  macro_f1  fit_time  score_time
0   KNN    256x192     1  0.459670  0.433599  1.194903    5.632386
1   KNN    256x192     2  0.501219  0.485314  1.165324    5.339069
2   KNN    256x192     3  0.603275  0.579804  1.186575    4.930135
3   KNN    256x192     4  0.558546  0.541348  1.220639    4.907500

Mean Accuracy: 0.5307
Accuracy Std: 0.0547

Mean Macro-F1: 0.5100
Macro-F1 Std: 0.0555

Total execution time: 25.61 seconds

CSV and JSON saved successfully!


[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:   25.6s finished


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path("..")

# Load training metadata
train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

# Load ONLY 256x192 HOG features
X_train = np.load(
    ROOT / "data/processed/driver_based/hog_256x192_train.npy",
    mmap_mode="r"
)

y_train = train_df["classname"].to_numpy()
groups_train = train_df["subject"].to_numpy()

# Restore the same 4-fold cross-validation
cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Verify dataset
assert X_train.shape == (15732, 25668)

print("Features:", X_train.shape)
print("Drivers:", len(np.unique(groups_train)))
print("Dataset ready for SVM!")

Features: (15732, 25668)
Drivers: 18
Dataset ready for SVM!


In [2]:
import time
import json
import os
import sys
import platform
import sklearn

from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate

# 1. Verify the dataset
assert X_train.shape == (15732, 25668)
assert len(X_train) == len(y_train) == len(groups_train)

# 2. Prepare the same 4 driver-independent folds
splits = list(cv.split(X_train, y_train, groups_train))

for train_idx, val_idx in splits:
    assert not (
        set(groups_train[train_idx]) &
        set(groups_train[val_idx])
    )

# 3. Create optimized Linear SVM
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        dual=False,
        max_iter=1000,
        tol=1e-2,
        random_state=42
    ))
])

# 4. Start timer
started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

# 5. Train 4 folds in parallel
svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    cv=splits,
    scoring=["accuracy", "f1_macro"],
    n_jobs=4,
    verbose=2,
    return_estimator=True,
    error_score="raise"
)

# 6. Record execution time
total_time = time.perf_counter() - start_time
completed_at = datetime.now(timezone.utc).isoformat()

# 7. Check optimization iterations
iterations = [
    int(model.named_steps["svm"].n_iter_)
    for model in svm_scores["estimator"]
]

iteration_limit_reached = [
    n >= 1000 for n in iterations
]

# 8. Save individual fold results
results_dir = ROOT / "results/classical/driver_based"
results_dir.mkdir(parents=True, exist_ok=True)

svm_results = pd.DataFrame({
    "model": "Linear SVM",
    "resolution": "256x192",
    "fold": range(1, 5),
    "accuracy": svm_scores["test_accuracy"],
    "macro_f1": svm_scores["test_f1_macro"],
    "fit_time": svm_scores["fit_time"],
    "score_time": svm_scores["score_time"],
    "iterations": iterations,
    "iteration_limit_reached": iteration_limit_reached
})

csv_path = results_dir / "linear_svm_256x192.csv"
svm_results.to_csv(csv_path, index=False)

# 9. Collect detailed fold information
fold_details = []

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train_drivers = set(groups_train[train_idx])
    val_drivers = set(groups_train[val_idx])

    fold_details.append({
        "fold": fold,
        "training_images": int(len(train_idx)),
        "validation_images": int(len(val_idx)),
        "training_drivers": sorted(train_drivers),
        "validation_drivers": sorted(val_drivers),
        "driver_overlap": len(train_drivers & val_drivers),
        "accuracy": float(svm_scores["test_accuracy"][fold - 1]),
        "macro_f1": float(svm_scores["test_f1_macro"][fold - 1]),
        "fit_time_seconds": float(svm_scores["fit_time"][fold - 1]),
        "score_time_seconds": float(svm_scores["score_time"][fold - 1]),
        "iterations": iterations[fold - 1],
        "iteration_limit_reached": iteration_limit_reached[fold - 1]
    })

# 10. Create complete JSON metadata
metadata = {
    "experiment": {
        "model": "Linear SVM",
        "resolution": "256x192",
        "feature_type": "HOG",
        "started_at": started_at,
        "completed_at": completed_at
    },

    "dataset": {
        "feature_file": "data/processed/driver_based/hog_256x192_train.npy",
        "training_images": int(X_train.shape[0]),
        "hog_features": int(X_train.shape[1]),
        "unique_drivers": int(len(np.unique(groups_train))),
        "classes": sorted(np.unique(y_train).tolist())
    },

    "model_configuration": {
        "pipeline": ["StandardScaler", "LinearSVC"],
        "scaler_parameters": svm_model.named_steps["scaler"].get_params(),
        "svm_parameters": svm_model.named_steps["svm"].get_params()
    },

    "cross_validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "parallel_workers": 4,
        "fold_details": fold_details
    },

    "results": {
        "mean_accuracy": float(svm_scores["test_accuracy"].mean()),
        "std_accuracy": float(svm_scores["test_accuracy"].std()),
        "mean_macro_f1": float(svm_scores["test_f1_macro"].mean()),
        "std_macro_f1": float(svm_scores["test_f1_macro"].std())
    },

    "timing": {
        "total_execution_seconds": float(total_time),
        "total_fit_time_seconds": float(svm_scores["fit_time"].sum()),
        "total_score_time_seconds": float(svm_scores["score_time"].sum())
    },

    "convergence": {
        "iterations_per_fold": iterations,
        "iteration_limit_reached": iteration_limit_reached,
        "convergence_verified": False,
        "note": "Review convergence warnings in notebook output."
    },

    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "operating_system": platform.platform(),
        "cpu_count": os.cpu_count()
    }
}

# 11. Save JSON
json_path = results_dir / "linear_svm_256x192_metadata.json"

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)

# 12. Display results
print("\nLinear SVM Results — 256x192")
print(svm_results)

print(f"\nMean Accuracy: {svm_scores['test_accuracy'].mean():.4f}")
print(f"Mean Macro-F1: {svm_scores['test_f1_macro'].mean():.4f}")
print(f"Total execution time: {total_time:.2f} seconds")

print("\nIterations per fold:", iterations)
print("Reached iteration limit:", iteration_limit_reached)

print("\nCSV and JSON saved successfully!")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END .................................................... total time= 8.9min
[CV] END .................................................... total time= 9.2min
[CV] END .................................................... total time= 9.3min
[CV] END .................................................... total time= 9.4min

Linear SVM Results — 256x192
        model resolution  fold  accuracy  macro_f1    fit_time  score_time  \
0  Linear SVM    256x192     1  0.419811  0.421614  531.078121    0.427905   
1  Linear SVM    256x192     2  0.538762  0.533823  550.903224    0.331573   
2  Linear SVM    256x192     3  0.656366  0.638001  557.236044    0.227224   
3  Linear SVM    256x192     4  0.592952  0.580369  566.275331    0.178353   

   iterations  iteration_limit_reached  
0           5                    False  
1           5                    False  
2           5                    False  
3           5                    False  

Mean Accuracy: 0.5520
Mean Macro-F1: 0.5435
Total

[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:  9.5min finished


FileNotFoundError: Missing experiment results: ../results/classical/driver_based/random_forest_128x96.csv

In [10]:
from pathlib import Path

ROOT = Path("..")

# Search for saved Random Forest results
for file in sorted((ROOT / "results").rglob("*random_forest*")):
    print(file)

../results/classical/driver_based/random_forest_256x192.csv
../results/classical/driver_based/random_forest_256x192_metadata.json
../results/classical/driver_based/random_forest_64x48.csv
../results/classical/driver_based/random_forest_64x48_metadata.json
../results/classical/driver_based/random_forest_resolution_comparison.json


In [11]:
import json

results_dir = ROOT / "results/classical/driver_based"

# Load existing Random Forest comparison
with open(results_dir / "random_forest_resolution_comparison.json") as f:
    rf_report = json.load(f)

# Check what information is available
print("Available sections:", list(rf_report.keys()))

if "experiments" in rf_report:
    print("Available resolutions:", list(rf_report["experiments"].keys()))

    if "128x96" in rf_report["experiments"]:
        print("\n128x96 experiment:")
        print(json.dumps(rf_report["experiments"]["128x96"], indent=2)[:3000])

Available sections: ['report_type', 'model', 'n_estimators', 'validation_strategy', 'dataset', 'comparison', 'best_resolution_by_mean_accuracy', 'best_resolution_by_mean_macro_f1', 'notes']


In [12]:
# Look for the 128x96 Random Forest experiment
for experiment in rf_report["comparison"]:
    if experiment.get("resolution") == "128x96":
        print(json.dumps(experiment, indent=4))

{
    "resolution": "128x96",
    "n_training_images": 15732,
    "hog_features": 5940,
    "model_configuration": {
        "estimator": "RandomForestClassifier",
        "explicit_kwargs": {
            "n_estimators": 500,
            "max_depth": null,
            "random_state": 42,
            "n_jobs": -1
        },
        "other_parameters": "scikit-learn defaults"
    },
    "cross_validation": {
        "strategy": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": true,
        "random_state": 42,
        "groups": "subject",
        "cv_parallel_jobs": 1
    },
    "fold_results": [
        {
            "fold": 1,
            "accuracy": 0.34245283,
            "macro_f1": 0.34124818,
            "fit_time_seconds": null,
            "score_time_seconds": null,
            "verbose_elapsed_minutes_approx": null
        },
        {
            "fold": 2,
            "accuracy": 0.61774744,
            "macro_f1": 0.59207357,
            "fit_time_seconds": 

In [13]:
import json
import pandas as pd
from pathlib import Path

ROOT = Path("..")
results_dir = ROOT / "results/classical/driver_based"

# Load existing comparison report
with open(results_dir / "random_forest_resolution_comparison.json") as f:
    rf_report = json.load(f)

# Extract ONLY 128x96 results
rf_128 = next(
    item for item in rf_report["comparison"]
    if item["resolution"] == "128x96"
)

# 1. Restore four-fold CSV
rf_results = pd.DataFrame(rf_128["fold_results"])

rf_results = rf_results.rename(columns={
    "fit_time_seconds": "fit_time",
    "score_time_seconds": "score_time"
})

rf_results.insert(0, "model", "Random Forest")
rf_results.insert(1, "resolution", "128x96")

assert len(rf_results) == 4
assert rf_128["hog_features"] == 5940

csv_path = results_dir / "random_forest_128x96.csv"
rf_results.to_csv(csv_path, index=False)

# 2. Restore detailed metadata
metadata = {
    "experiment": {
        "model": "Random Forest",
        "resolution": "128x96",
        "feature_type": "HOG",
        "status": "recovered_from_previous_results"
    },

    "dataset": {
        "training_images": rf_128["n_training_images"],
        "hog_features": rf_128["hog_features"]
    },

    "model_configuration": rf_128["model_configuration"],

    "cross_validation": rf_128["cross_validation"],

    "results": {
        "mean_accuracy": float(rf_results["accuracy"].mean()),
        "std_accuracy": float(rf_results["accuracy"].std(ddof=0)),
        "mean_macro_f1": float(rf_results["macro_f1"].mean()),
        "std_macro_f1": float(rf_results["macro_f1"].std(ddof=0))
    },

    "timing": {
        "total_execution_seconds": rf_128["timing"][
            "cross_validation_wall_time_seconds"
        ],
        "precision": "approximate",
        "per_fold_times": "unavailable"
    },

    "provenance": rf_128["provenance"]
}

json_path = results_dir / "random_forest_128x96_metadata.json"

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)

# Verify
print("Recovered Random Forest results:")
print(rf_results[["fold", "accuracy", "macro_f1"]])

print("\nCSV saved:", csv_path)
print("JSON saved:", json_path)

Recovered Random Forest results:
   fold  accuracy  macro_f1
0     1  0.342453  0.341248
1     2  0.617747  0.592074
2     3  0.628104  0.603810
3     4  0.633463  0.619353

CSV saved: ../results/classical/driver_based/random_forest_128x96.csv
JSON saved: ../results/classical/driver_based/random_forest_128x96_metadata.json
